# Artisys V2 — animar imagem no Colab grátis

Use uma imagem definida como referência do personagem, para preservar melhor os dois olhos, braços e pernas. **Wan 2.2 TI2V-5B (Apache 2.0)** roda por uma API local do ComfyUI dentro do Colab, sem expor servidor público, sem Cloudflare ou serviço pago.

**iPhone:** ative GPU no ambiente de execução; execute as células em ordem; na segunda, envie a foto PNG/JPG/WebP do personagem; personalize o movimento; gere; baixe o MP4.

**Limites:** pesos de cerca de 17 GB, armazenamento temporário e memória RAM; T4 gratuita pode ser lenta, encerrar a sessão ou não comportar a geração. Não há garantia de qualidade nem execução bem-sucedida. Não ativa GPU paga automaticamente. Se faltar recurso, interrompa e tente um teste mais leve. Não substitui o notebook V1.


In [ ]:
# @title 1. Verificar GPU e armazenamento antes dos downloads
import torch
import shutil
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Sem GPU gratuita disponível. Ambiente de execução > Alterar tipo > GPU. Não contrate nada.")
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
free = shutil.disk_usage("/content").free / 1024**3
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(vram,1), "| Disco GiB:", round(free,1))
if vram < 12:
    raise RuntimeError("GPU com menos de 12 GiB. É recomendado aguardar uma T4 gratuita ou maior.")
if free < 23:
    raise RuntimeError("São necessários cerca de 23 GiB de disco livre, incluindo os modelos.")


In [ ]:
# @title 2. Enviar referência: personagem com dois olhos e membros visíveis
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
from io import BytesIO
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Envie uma única imagem.")
filename, data = next(iter(uploaded.items()))
if len(data) > 12_000_000 or not filename.lower().endswith((".png",".jpg",".jpeg",".webp")):
    raise ValueError("Envie PNG/JPG/WebP de até 12 MB.")
try:
    picture = ImageOps.exif_transpose(Image.open(BytesIO(data))).convert("RGB")
except Exception as exc:
    raise ValueError("Não foi possível ler a imagem.") from exc
if min(picture.size) < 320:
    raise ValueError("Use uma imagem de pelo menos 320 px no menor lado.")
work = Path("/content/artisys-v2")
work.mkdir(parents=True, exist_ok=True)
reference = ImageOps.pad(picture, (448,800), Image.Resampling.LANCZOS, color=(26,27,51))
reference.save(work / "personagem.png")
display(reference.resize((224,400)))
print("Referência pronta. Braços, pernas e olhos não foram cortados.")


In [ ]:
# @title 3. Instalar ComfyUI e baixar Wan 2.2 (aprox. 17 GB)
from pathlib import Path
import subprocess, sys
root = Path("/content/ComfyUI")
if not root.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",str(root)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(root/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub>=0.31,<2"],check=True)
from huggingface_hub import hf_hub_download
specs = [
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", "diffusion_models"),
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders"),
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors", "vae")
]
for repo_id, remote, category in specs:
    local_dir = root/"models"/category
    local_dir.mkdir(parents=True, exist_ok=True)
    target = local_dir/Path(remote).name
    if target.exists() and target.stat().st_size > 100_000_000:
        print("Reutilizando:", target.name)
        continue
    source = Path(hf_hub_download(repo_id=repo_id,filename=remote))
    if target.is_symlink() or target.exists():
        target.unlink()
    target.symlink_to(source)
    print("Modelo pronto:", target.name)
print("Dependências e modelos disponíveis. Sem API de cobrança.")


In [ ]:
# @title 4. Ajustar movimento e resolução
movimento = "An adorable pink brain character with exactly two large blue eyes, two arms and two legs gently raises one hand in a slow friendly wave. Fixed camera, subtle smiling expression, stable anatomy, stable background, glowing neural connections, polished stylized 3D educational cartoon. Keep the original design." # @param {type:"string"}
perfil = "T4 conservador" # @param ["T4 conservador", "T4 qualidade maior"]
quadros = 41 # @param {type:"integer"}
passos = 20 # @param {type:"integer"}
semente = 42 # @param {type:"integer"}
fps = 8 # @param {type:"integer"}
if not 5 <= len(movimento) <= 3000:
    raise ValueError("Descreva o movimento com 5 a 3000 caracteres.")
if quadros not in (17,25,33,41,49):
    raise ValueError("Quadros válidos: 17, 25, 33, 41 ou 49.")
if not 12 <= passos <= 35 or not 4 <= fps <= 24 or not 0 <= semente < 2**32:
    raise ValueError("Verifique passos (12-35), FPS (4-24) e semente.")
width,height = (352,640) if perfil=="T4 conservador" else (448,800)
print("Resolução:",width,"x",height,"| Quadros:",quadros,"| Duração aproximada:",round(quadros/fps,1),"s")


In [ ]:
# @title 5. Animar imagem com Wan 2.2 via API local do ComfyUI
import requests
import subprocess
import time
import shutil
import sys
from pathlib import Path
from PIL import Image, ImageOps
root=Path("/content/ComfyUI")
work=Path("/content/artisys-v2")
if not (work/"personagem.png").exists():
    raise FileNotFoundError("Execute a célula 2.")
for name,kind in [("wan2.2_ti2v_5B_fp16.safetensors","diffusion_models"),("umt5_xxl_fp8_e4m3fn_scaled.safetensors","text_encoders"),("wan2.2_vae.safetensors","vae")]:
    if not (root/"models"/kind/name).exists():
        raise FileNotFoundError("Execute célula 3: "+name)
im=Image.open(work/"personagem.png")
ImageOps.pad(im,(width,height),Image.Resampling.LANCZOS,color=(26,27,51)).save(root/"input"/"artisys_v2.png")
log=work/"comfyui.log"
proc=globals().get("_artisys_proc")
if proc is None or proc.poll() is not None:
    loghandle=open(log,"a",encoding="utf-8")
    proc=subprocess.Popen([sys.executable,"main.py","--listen","127.0.0.1","--port","8188","--lowvram","--disable-auto-launch"],cwd=str(root),stdout=loghandle,stderr=subprocess.STDOUT)
    globals()["_artisys_proc"]=proc
base="http://127.0.0.1:8188"
for _ in range(90):
    if proc.poll() is not None:
        raise RuntimeError("ComfyUI encerrou. Confira "+str(log))
    try:
        if requests.get(base+"/system_stats",timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError("ComfyUI não iniciou em 3 minutos. Confira o log.")
required={"UNETLoader","CLIPLoader","VAELoader","LoadImage","CLIPTextEncode","Wan22ImageToVideoLatent","ModelSamplingSD3","KSampler","VAEDecode","CreateVideo","SaveVideo"}
nodes=set(requests.get(base+"/object_info",timeout=30).json())
if required-nodes:
    raise RuntimeError("Nós do ComfyUI ausentes: "+str(sorted(required-nodes)))
negative="extra eyes, single eye, third eye, extra arms, fused arms, extra legs, missing limbs, deformed face, morphing, shaky camera, text, watermark, flicker"
workflow={
"1":{"class_type":"UNETLoader","inputs":{"unet_name":"wan2.2_ti2v_5B_fp16.safetensors","weight_dtype":"default"}},
"2":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan","device":"default"}},
"3":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
"4":{"class_type":"LoadImage","inputs":{"image":"artisys_v2.png"}},
"5":{"class_type":"CLIPTextEncode","inputs":{"clip":["2",0],"text":movimento}},
"6":{"class_type":"CLIPTextEncode","inputs":{"clip":["2",0],"text":negative}},
"7":{"class_type":"Wan22ImageToVideoLatent","inputs":{"vae":["3",0],"start_image":["4",0],"width":width,"height":height,"length":quadros,"batch_size":1}},
"8":{"class_type":"ModelSamplingSD3","inputs":{"model":["1",0],"shift":8}},
"9":{"class_type":"KSampler","inputs":{"model":["8",0],"positive":["5",0],"negative":["6",0],"latent_image":["7",0],"seed":semente,"steps":passos,"cfg":5,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0}},
"10":{"class_type":"VAEDecode","inputs":{"samples":["9",0],"vae":["3",0]}},
"11":{"class_type":"CreateVideo","inputs":{"images":["10",0],"fps":fps}},
"12":{"class_type":"SaveVideo","inputs":{"video":["11",0],"filename_prefix":"Artisys/brain_v2","format":"auto","codec":"auto"}}
}
reply=requests.post(base+"/prompt",json={"prompt":workflow},timeout=120)
if not reply.ok:
    raise RuntimeError("O ComfyUI rejeitou o workflow: "+reply.text[:1500])
job_data=reply.json()
if job_data.get("node_errors"):
    raise RuntimeError("Erros nos nós: "+str(job_data["node_errors"])[:1500])
job_id=job_data["prompt_id"]
print("Iniciando job:",job_id,"— aguarde; pode demorar bastante na T4.")
begin=time.monotonic()
while time.monotonic()-begin < 7200:
    if proc.poll() is not None:
        raise RuntimeError("ComfyUI encerrou. Confira "+str(log))
    history=requests.get(base+"/history/"+job_id,timeout=30).json()
    job=history.get(job_id)
    if job:
        status=job.get("status",{})
        if status.get("status_str")=="error":
            raise RuntimeError("Falha ao gerar. Confira "+str(log)+" "+str(status.get("messages",[]))[-1000:])
        if status.get("completed") or status.get("status_str")=="success":
            break
    time.sleep(15)
else:
    raise TimeoutError("O job excedeu 2 horas. Verifique o log antes de reenviar.")
files=sorted((root/"output"/"Artisys").glob("brain_v2_*.mp4"),key=lambda p:p.stat().st_mtime,reverse=True)
if not files:
    raise FileNotFoundError("Nenhum MP4 encontrado. Confira "+str(log))
final=work/"cena_01_v2.mp4"
shutil.copy2(files[0],final)
print("MP4 final:",final,"| tempo de geração:",round((time.monotonic()-begin)/60,1),"min")


In [ ]:
# @title 6. Assistir, baixar ou salvar no Drive
from pathlib import Path
from IPython.display import Video, display
video=Path("/content/artisys-v2/cena_01_v2.mp4")
if not video.exists():
    raise FileNotFoundError("Execute a célula 5 até ela mostrar MP4 final.")
display(Video(str(video),embed=True,width=400))
baixar_agora = False # @param {type:"boolean"}
salvar_no_drive = False # @param {type:"boolean"}
if baixar_agora:
    from google.colab import files
    files.download(str(video))
if salvar_no_drive:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destino=Path("/content/drive/MyDrive/ArtisysVideos")
    destino.mkdir(parents=True,exist_ok=True)
    shutil.copy2(video,destino/video.name)
    print("Salvo no Drive:",destino/video.name)
